# Offline language-ID deployment demo

Load the three fifteen-language deployment models directly from extracted folders:

| Notebook key | Deployment folder | Adaptation |
|---|---|---|
| `whisper_small_frozen` | `whisper_small15_frozen` | Classifier trained; encoder frozen |
| `speechbrain_partial` | `lid15_partial` | Encoder tail and classifier trained |
| `whisper_tiny_frozen` | `whisper_tiny15_frozen` | Classifier trained; encoder frozen |

This notebook shows the **supported language set**, **single-file inference**, and
**batched inference with unequal audio lengths**. It uses no `lid` project imports,
training checkpoints, dataset downloads, API credentials, or internet services.
All three use SpeechBrain's `EncoderClassifier` interface.

## Before going offline

- Extract each deployment archive and keep its complete folder, including the
  Whisper bundles' `custom.py` files.
- Have a Python/Jupyter kernel with `torch`, `torchaudio`, `speechbrain`,
  `openai-whisper` and `soundfile` already installed. Tested model versions are
  recorded in each bundle's `export.json`.
- In this repository, `uv sync --group notebook` prepares notebook tooling **before**
  disconnecting. Select `.venv/bin/python` as the notebook kernel in your editor.
  Outside this repository, prepare the same dependencies in your own notebook environment.
- Place your audio on disk. WAV files are used below. Defaults select three existing
  v4 validation segments when this notebook is opened in this repository; otherwise
  set `AUDIO_FILES` to your own local files.

There are no install or download cells. CPU is the default. Run the cells in order.


In [ ]:
import os

# Set offline behavior before importing the inference libraries.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "2"

import importlib.metadata
import importlib.util
import json
import sys
import tempfile
from contextlib import contextmanager
from pathlib import Path

import soundfile as sf
import torch
import torchaudio
from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.fetching import LocalStrategy

torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

DEVICE = "cpu"  # Change to "cuda" if desired and available.
SAMPLE_RATE = 16_000
MAX_SECONDS = 10
BATCH_SIZE = 2

workspace = tempfile.TemporaryDirectory(prefix="offline-lid-demo-")
WORK_DIR = Path(workspace.name)  # Temporary files; deployment folders stay unchanged.

for package in ("torch", "torchaudio", "speechbrain", "openai-whisper", "soundfile"):
    print(f"{package}: {importlib.metadata.version(package)}")
print(f"Device: {DEVICE}")


## 1. Choose local deployment folders and audio

Edit `DEPLOYMENT_DIR` and `AUDIO_FILES` if you have copied the notebook elsewhere.
Paths must refer to **existing local files and folders**, not Hub names or URLs.
You can select a subset of the three models with `ENABLED_MODELS`.


In [ ]:
# Locate the repository from either its root or the notebooks/ directory.
# When using a different directory layout, replace DEPLOYMENT_DIR explicitly.
PROJECT_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "artifacts" / "deployment").is_dir()),
    Path.cwd(),
)
DEPLOYMENT_DIR = PROJECT_DIR / "artifacts" / "deployment"
# Example elsewhere: DEPLOYMENT_DIR = Path("/path/to/extracted/models")

MODEL_BUNDLES = {
    "whisper_small_frozen": DEPLOYMENT_DIR / "whisper_small15_frozen",
    "speechbrain_partial": DEPLOYMENT_DIR / "lid15_partial",
    "whisper_tiny_frozen": DEPLOYMENT_DIR / "whisper_tiny15_frozen",
}
ENABLED_MODELS = list(MODEL_BUNDLES)

# Once the Large v3 experiment has produced its verified local export,
# uncomment these lines to run that model through the same examples:
# MODEL_BUNDLES["whisper_large_v3_frozen"] = DEPLOYMENT_DIR / "whisper_large_v3_15_frozen"
# ENABLED_MODELS = ["whisper_large_v3_frozen"]
# Large v3 uses more memory; start with that model alone on CPU.

SEGMENT_DIR = PROJECT_DIR / "artifacts" / "youtube_validation_v4" / "segments"
AUDIO_FILES = []
for prefix in ("sam", "niu", "ton"):
    matches = sorted(SEGMENT_DIR.glob(f"{prefix}-*.wav"))
    if matches:
        AUDIO_FILES.append(matches[0].resolve())

# Replace defaults with your own files, for example:
# AUDIO_FILES = [Path("/path/to/first.wav"), Path("/path/to/second.wav")]

if not ENABLED_MODELS:
    raise ValueError("Choose at least one model in ENABLED_MODELS.")
for key in ENABLED_MODELS:
    if not MODEL_BUNDLES[key].is_dir():
        raise FileNotFoundError(f"Extract the local deployment folder: {MODEL_BUNDLES[key]}")
if not AUDIO_FILES:
    raise FileNotFoundError("Set AUDIO_FILES above to one or more local audio files.")
AUDIO_FILES = [Path(path).expanduser().resolve(strict=True) for path in AUDIO_FILES]

print("Deployment root:", DEPLOYMENT_DIR)
print("Models:", ", ".join(ENABLED_MODELS))
for path in AUDIO_FILES:
    print("Audio:", path.name)


## 2. Load models from disk

`from_hparams(source=...)` receives an absolute local folder. `NO_LINK` reads the
existing weight files directly, without downloading or copying model weights.
The loader below also selects the correct bundled `custom.py` while constructing
each Whisper model, so older Tiny and newer Small exports can coexist in one kernel.

**Calibration is already folded into these classifiers.** Their outputs are log
probabilities: use `.exp()` to get probabilities. Do not apply a second temperature
or a second softmax.


In [ ]:
@contextmanager
def bundled_backend(bundle):
    """Use this bundle's custom backend during model construction only."""
    custom_file = bundle / "custom.py"
    previous_module = sys.modules.get("custom")
    previous_path = sys.path.copy()
    try:
        if custom_file.is_file():
            spec = importlib.util.spec_from_file_location("custom", custom_file)
            module = importlib.util.module_from_spec(spec)
            sys.modules["custom"] = module
            spec.loader.exec_module(module)
        yield
    finally:
        sys.path[:] = previous_path
        if previous_module is None:
            sys.modules.pop("custom", None)
        else:
            sys.modules["custom"] = previous_module


def load_local_model(key):
    bundle = MODEL_BUNDLES[key].expanduser().resolve(strict=True)
    required = ["hyperparams.yaml", "embedding_model.ckpt", "classifier.ckpt",
                "label_encoder.ckpt", "export.json"]
    if "custom." in (bundle / "hyperparams.yaml").read_text():
        required.append("custom.py")
    missing = [name for name in required if not (bundle / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Incomplete deployment folder {bundle}: {missing}")

    with bundled_backend(bundle):
        classifier = EncoderClassifier.from_hparams(
            source=str(bundle),
            savedir=str(WORK_DIR / key),
            local_strategy=LocalStrategy.NO_LINK,
            run_opts={"device": DEVICE},
        )
    classifier.eval()
    metadata = json.loads((bundle / "export.json").read_text())
    classifier.hparams.label_encoder.expect_len(len(metadata["codes"]))
    return classifier


In [ ]:
models = {key: load_local_model(key) for key in ENABLED_MODELS}
print("Loaded from local deployment artifacts:", ", ".join(models))


## 3. Read the supported languages from each loaded model

The label encoder gives the exact classifier output order. The supported set comes
from the loaded model, rather than a hard-coded list of languages.


In [ ]:
def supported_languages(classifier):
    encoder = classifier.hparams.label_encoder
    labels = encoder.decode_torch(torch.arange(len(encoder)))
    result = []
    for index, label in enumerate(labels):
        code, _, name = label.partition(":")
        result.append({"index": index, "code": code.strip(), "name": name.strip()})
    return result


languages = {key: supported_languages(classifier) for key, classifier in models.items()}
supported_codes = {key: {row["code"] for row in entries} for key, entries in languages.items()}

for key, codes in supported_codes.items():
    print(f"{key}: {len(codes)} languages: {', '.join(sorted(codes))}")
    metadata = json.loads((MODEL_BUNDLES[key] / "export.json").read_text())
    assert [row["code"] for row in languages[key]] == metadata["codes"]

first_model_key = next(iter(models))
print(f"\nClassifier output order for {first_model_key}:")
for row in languages[first_model_key]:
    print(f"{row['index']:2d}  {row['code']:3s}  {row['name']}")


## 4. Prepare audio consistently

For these deployment bundles, use mono 16 kHz speech and at most ten real seconds.
The helper resamples, mixes channels, then center-crops long audio. It rejects
empty, non-finite or shorter-than-0.1-second inputs. It does not perform VAD.

Whisper's custom backend adds its native thirty-second padding internally.
SpeechBrain's `classify_file` does not apply the ten-second crop itself, so the
single-file example writes the prepared waveform to a temporary WAV first.


In [ ]:
def prepare_audio(path):
    path = Path(path).expanduser().resolve(strict=True)
    waveform, sample_rate = torchaudio.load(str(path), backend="soundfile")
    if waveform.ndim != 2 or waveform.shape[0] == 0 or waveform.shape[1] == 0:
        raise ValueError(f"Empty or invalid audio: {path}")
    waveform = waveform.mean(dim=0).float()  # [samples], mono
    if not torch.isfinite(waveform).all():
        raise ValueError(f"Non-finite audio: {path}")
    if sample_rate != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)
    size = min(len(waveform), MAX_SECONDS * SAMPLE_RATE)
    start = (len(waveform) - size) // 2
    waveform = waveform[start:start + size].contiguous()
    if len(waveform) < SAMPLE_RATE // 10:
        raise ValueError(f"Audio must contain at least 0.1 seconds: {path}")
    return waveform


def unpack_predictions(key, output, paths):
    log_probs, log_score, class_index, text_labels = output
    probabilities = log_probs.detach().cpu().exp()
    assert probabilities.shape == (len(paths), len(languages[key]))
    torch.testing.assert_close(
        probabilities.sum(dim=1), torch.ones(len(paths)), atol=1e-5, rtol=1e-5
    )
    predictions = []
    for row, path in enumerate(paths):
        index = int(class_index[row])
        label = languages[key][index]
        predictions.append({
            "model": key,
            "file": str(path),
            "code": label["code"],
            "language": label["name"],
            "confidence": float(probabilities[row, index]),
            "probabilities": {entry["code"]: float(probabilities[row, entry["index"]])
                              for entry in languages[key]},
        })
    return predictions


def show_predictions(predictions):
    for item in predictions:
        print(f"{item['model']:24s}  {Path(item['file']).name[:30]:30s}  "
              f"{item['code']:3s}  {item['language']:17s}  {item['confidence']:.2%}")


## 5. Single-file inference

All three models receive exactly the same prepared local audio. The returned
confidence is the exported classifier's probability of its top prediction.


In [ ]:
single_path = AUDIO_FILES[0]
single_waveform = prepare_audio(single_path)
prepared_single_path = WORK_DIR / "single_16khz_mono.wav"
sf.write(str(prepared_single_path), single_waveform.numpy(), SAMPLE_RATE, subtype="FLOAT")

single_predictions = []
with torch.inference_mode():
    for key, classifier in models.items():
        output = classifier.classify_file(str(prepared_single_path))
        single_predictions.extend(unpack_predictions(key, output, [single_path]))

print(f"Single input: {len(single_waveform) / SAMPLE_RATE:.3f} seconds, mono {SAMPLE_RATE} Hz")
show_predictions(single_predictions)


You can also inspect every supported output probability. For example, the top
five predictions from the first model:


In [ ]:
top_five = sorted(single_predictions[0]["probabilities"].items(), key=lambda item: item[1], reverse=True)[:5]
for language_code, probability in top_five:
    print(f"{language_code:3s}  {probability:.2%}")


## 6. Batch inference, including unequal lengths

Prepare each file independently, pad waveforms on the right, and pass `wav_lens`:
each example's real sample count divided by the padded batch length. This keeps
padding separate from real audio. Do not leave out lengths for unequal inputs.

The helper processes files in small batches and returns predictions in the same
order as the input paths. Some SpeechBrain scores can vary slightly with batch
padding; compare predictions without assuming bit-for-bit score equality.


In [ ]:
@torch.inference_mode()
def predict_batch(key, paths, batch_size=BATCH_SIZE):
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    paths = [Path(path).expanduser().resolve(strict=True) for path in paths]
    results = []
    for start in range(0, len(paths), batch_size):
        selected = paths[start:start + batch_size]
        waveforms = [prepare_audio(path) for path in selected]
        sample_counts = torch.tensor([len(waveform) for waveform in waveforms])
        padded = torch.nn.utils.rnn.pad_sequence(waveforms, batch_first=True)
        wav_lens = sample_counts.float() / padded.shape[1]
        print(f"{key}: padded shape={tuple(padded.shape)}, relative lengths={wav_lens.tolist()}")
        output = models[key].classify_batch(padded, wav_lens=wav_lens)
        results.extend(unpack_predictions(key, output, selected))
    return results


batch_predictions = {}
for key in models:
    batch_predictions[key] = predict_batch(key, AUDIO_FILES)
    show_predictions(batch_predictions[key])
    print()


## Using your own audio

Change `AUDIO_FILES` and rerun the single/batch sections. To use another evaluated
deployment variant, change its folder in `MODEL_BUNDLES` and reload that model.
You can use the same functions outside a notebook: no repository modules are required.

These are closed-set language-ID models, not transcription models. They always
choose from their supported language set. Prediction confidence alone is not an
unknown-language or English/code-switch annotation. Example predictions here are
an API demonstration, not an accuracy evaluation.
